### page6.chapter1.[실습] Tool 여러 개 준비하기

In [ ]:
from langchain_core.tools import tool
import ast, operator

# [1번 문제]
# 도시 이름을 받아 날씨를 반환하는 get_weather(city: str) 함수를
# @tool 데코레이터와 함께 작성하세요. (docstring 필수!)
@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 날씨 정보를 반환합니다."""
    weather_db = {"서울": "맑음, 23도"}
    return weather_db.get(city, "정보 없음")

# [2번 문제]
# 사칙연산 문자열(예: "3 + 5")을 계산해 반환하는
# calculate(expression: str) 함수를 @tool로 작성하세요.
# (ast.parse와 ast.BinOp를 활용하면 eval보다 안전하게 계산할 수 있습니다)
# eval("5+6") --> 파이썬 코드로 실행해줌
_OPS = {
    ast.Add: operator.add,      # + 기호
    ast.Sub: operator.sub,      # - 기호
    ast.Mult: operator.mul,     # * 기호
    ast.Div: operator.truediv,  # / 기호
    ast.USub: operator.neg,     # 음수 부호 (-)
    ast.UAdd: operator.pos      # 양수 부호 (+)
}
def _safe_eval(node): # 함수명 앞에 _를 붙인건 private의 의미임
    if isinstance(node, ast.Constant):# 숫자인지?
        return node.value
    if isinstance(node, ast.BinOp):# Binary Operation 이항연산자인가?
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):# 단항연산자인가?
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("허용되지 않은 수식입니다.")

@tool
def calculate(expression: str) -> str:
    """사칙연산 수식 문자열을 계산합니다. 예: '3 + 5 * 2'"""
    try:# mode를 eval로 지정하면 expression문자열에 단일 수식만 허용하고 
        tree = ast.parse(expression, mode="eval") # 결과물은 Tree 객체로 반환(연산자 우선순위로 정리된 계층구조)
        result = _safe_eval(tree.body) # Tree 안에 값을 꺼내서 함수로 전달
        return f"{expression} = {result}"
    except Exception:
        return "계산할 수 없는 수식입니다."
# [도전 문제]
# get_weather.invoke({"city": "부산"}) 와
# calculate.invoke({"expression": "10 * (2 + 3)"}) 를 각각 실행해
# Tool 함수가 Agent 없이도 단독으로 잘 동작하는지 먼저 확인하세요.
